# 결과 표 — 축정렬 5방법 × 2모델 스윕 (2026-07-22)

`results/**/*.json` 의 multi-seed 결과 파일을 전부 읽어서 표로 찍는다.

**규약** (`protocol-train-val-test-split`): 세션 단위 층화 7:1.5:1.5 분할.
model selection 은 target **val**(oracle), 보고 수치는 target **TEST**.
`Tgt-Val − Tgt-Test` 는 selection 때문에 생긴 낙관 편향의 크기다.

test 지표가 없는 파일은 옛 2분할 규약으로 돈 것이라 `protocol` 컬럼에서 구분되고,
메인 표에서는 제외된다 (같은 표에 섞으면 비교 불가).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# notebook/ 에서 실행하든 repo root 에서 실행하든 results/ 를 찾아 올라간다
ROOT = Path.cwd()
while not (ROOT / "results").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
assert RESULTS.is_dir(), f"results/ 를 못 찾음 (cwd={Path.cwd()})"

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 200)

# json metric key -> 표 컬럼명
METRICS = {
    "source_acc":      "Src-Val",
    "target_acc":      "Tgt-Val",
    "shift":           "Shift-Val",
    "source_test_acc": "Src-Test",
    "target_test_acc": "Tgt-Test",
    "test_shift":      "Shift-Test",
}
# 축정렬 방법 (표 정렬 순서 = 정렬 강도 순, raw 가 무처리)
METHOD_ORDER = ["raw", "permutation", "gravity", "kabsch", "pca"]

print("repo root :", ROOT)

repo root : /home/user1/Domain_Adaptation_for_EMG_IMU_Sensor


## 1. 로더 — 모든 multi-seed 결과 json 을 seed 단위 long-form 으로

In [2]:
def load_runs(results_dir=RESULTS):
    """results/ 아래 multi-seed 스키마(results/mean/std 키를 가진) json 을 전부 읽어
    '한 행 = 한 seed' 인 데이터프레임으로 만든다. 다른 스키마 파일은 조용히 건너뛴다."""
    rows = []
    for p in sorted(Path(results_dir).rglob("*.json")):
        try:
            j = json.loads(p.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue
        runs = j.get("results")
        if not isinstance(runs, list) or not runs or "mean" not in j:
            continue                     # multi-seed 결과 파일이 아님
        for r in runs:
            if not isinstance(r, dict):
                continue
            row = {
                "group":     p.parent.name,                  # IMU / Multimodal / ...
                "file":      p.relative_to(ROOT).as_posix(),
                "tag":       j.get("tag"),
                "modality":  j.get("modality"),
                "model":     j.get("model") or j.get("mode"),
                "data_dir":  j.get("data_dir"),
                "selection": j.get("selection"),
                "mtime":     pd.Timestamp(p.stat().st_mtime, unit="s", tz="UTC")
                               .tz_convert("Asia/Seoul").strftime("%Y-%m-%d %H:%M"),
                "seed":      r.get("seed"),
            }
            row.update({k: r.get(k) for k in METRICS})
            rows.append(row)

    df = pd.DataFrame(rows)
    # test 지표 유무가 곧 규약 구분이다
    df["protocol"] = np.where(df["target_test_acc"].notna(),
                              "train/val/test", "train/val (구)")
    return df


df_runs = load_runs()
print(f"{len(df_runs)} runs / {df_runs['file'].nunique()} result files")
df_runs.head()

305 runs / 61 result files


,group,file,tag,modality,model,data_dir,selection,mtime,seed,source_acc,target_acc,shift,source_test_acc,target_test_acc,test_shift,protocol
0,IMU,results/IMU/imu_cdan_result_gravity.json,gravity,imu_only,None,preprocessed_MM_gravity,target_val (oracle),2026-07-22 16:08,0,92.701950,79.378440,13.323510,92.622081,77.451556,15.170525,train/val/test
1,IMU,results/IMU/imu_cdan_result_gravity.json,gravity,imu_only,None,preprocessed_MM_gravity,target_val (oracle),2026-07-22 16:08,1,84.791086,70.346390,14.444696,81.422505,64.092777,17.329728,train/val/test
2,IMU,results/IMU/imu_cdan_result_gravity.json,gravity,imu_only,None,preprocessed_MM_gravity,target_val (oracle),2026-07-22 16:08,2,91.030641,76.561994,14.468646,88.694268,75.249560,13.444708,train/val/test
3,IMU,results/IMU/imu_cdan_result_gravity.json,gravity,imu_only,None,preprocessed_MM_gravity,target_val (oracle),2026-07-22 16:08,3,90.807799,74.554872,16.252927,88.375796,69.142689,19.233107,train/val/test
4,IMU,results/IMU/imu_cdan_result_gravity.json,gravity,imu_only,None,preprocessed_MM_gravity,target_val (oracle),2026-07-22 16:08,4,93.203343,68.112658,25.090685,89.225053,64.327657,24.897396,train/val/test


In [3]:
def agg_pm(df, keys, metrics=None, nd=2):
    """keys 로 묶어 각 지표를 'mean ± std' 문자열 한 칸으로 만든 보기용 표."""
    metrics = list(metrics or METRICS)
    g = df.groupby(keys, dropna=False, observed=True)
    out = pd.DataFrame({"n": g.size()})
    for m in metrics:
        mu, sd = g[m].mean(), g[m].std()
        out[METRICS[m]] = [
            "—" if pd.isna(a) else (f"{a:.{nd}f} ± {b:.{nd}f}" if pd.notna(b) else f"{a:.{nd}f}")
            for a, b in zip(mu, sd)
        ]
    return out.reset_index()


def agg_num(df, keys, metrics=None):
    """숫자 그대로 필요한 경우 (정렬·차이 계산·플롯용). 컬럼 = (지표, mean|std)."""
    metrics = list(metrics or METRICS)
    g = df.groupby(keys, dropna=False, observed=True)
    out = g[metrics].agg(["mean", "std"]).round(2)
    out.insert(0, ("n", ""), g.size())
    return out


def to_markdown(df, floatfmt="{:.2f}"):
    """tabulate 없이 마크다운 표 문자열 생성 (README/보고서에 붙여넣기용)."""
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else floatfmt.format(v))
    cols = [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c) for c in d.columns]
    lines = ["| " + " | ".join(cols) + " |",
             "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in d.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |")
    return "\n".join(lines)

## 2. 메인 표 — 2026-07-22 스윕

`raw / permutation / gravity / kabsch / pca` × `IMU 단독 / 멀티모달`, seed 0–4.

In [4]:
SWEEP = {                                   # 표시 이름 -> 결과 파일 패턴
    "IMU 단독": "results/IMU/imu_cdan_result_{}.json",
    "멀티모달":  "results/Multimodal/cdan_result_{}.json",
}
lookup = {tmpl.format(m): (label, m)
          for label, tmpl in SWEEP.items() for m in METHOD_ORDER}

sweep = df_runs[df_runs.file.isin(lookup)].copy()
sweep["model"]  = pd.Categorical(sweep.file.map(lambda f: lookup[f][0]), list(SWEEP), ordered=True)
sweep["method"] = pd.Categorical(sweep.file.map(lambda f: lookup[f][1]), METHOD_ORDER, ordered=True)

missing = set(lookup) - set(sweep.file)
if missing:
    print("!! 결과 파일 없음:", *sorted(missing), sep="\n   ")
assert (sweep.protocol == "train/val/test").all(), "옛 2분할 규약 파일이 섞였다 — 재학습 필요"
print(f"{sweep.method.nunique()} methods × {sweep.model.nunique()} models × "
      f"{sweep.seed.nunique()} seeds = {len(sweep)} runs")

5 methods × 2 models × 5 seeds = 50 runs


### 2-1. 전체 지표 (mean ± std, %)

In [5]:
tbl_full = agg_pm(sweep, ["model", "method"])
tbl_full

,model,method,n,Src-Val,Tgt-Val,Shift-Val,Src-Test,Tgt-Test,Shift-Test
0,IMU 단독,raw,5,87.72 ± 2.55,21.36 ± 2.37,66.36 ± 3.64,82.01 ± 4.16,21.81 ± 2.73,60.19 ± 4.67
1,IMU 단독,permutation,5,89.83 ± 2.87,75.85 ± 2.68,13.98 ± 4.12,88.13 ± 2.54,74.22 ± 2.30,13.92 ± 3.12
2,IMU 단독,gravity,5,90.51 ± 3.36,73.79 ± 4.57,16.72 ± 4.80,88.07 ± 4.08,70.05 ± 6.14,18.02 ± 4.42
3,IMU 단독,kabsch,5,91.11 ± 1.98,62.64 ± 4.34,28.47 ± 5.19,88.79 ± 0.85,63.78 ± 5.13,25.01 ± 4.79
4,IMU 단독,pca,5,93.64 ± 1.90,75.36 ± 3.06,18.27 ± 3.93,90.66 ± 1.97,75.00 ± 2.01,15.66 ± 1.52
5,멀티모달,raw,5,91.30 ± 3.73,35.29 ± 3.69,56.01 ± 3.07,87.78 ± 3.49,31.76 ± 1.73,56.03 ± 2.89
6,멀티모달,permutation,5,95.47 ± 2.22,89.91 ± 0.81,5.56 ± 1.46,92.26 ± 1.25,87.45 ± 2.63,4.82 ± 1.78
7,멀티모달,gravity,5,95.26 ± 1.82,91.97 ± 1.68,3.29 ± 2.50,92.90 ± 0.94,88.75 ± 1.98,4.15 ± 1.51
8,멀티모달,kabsch,5,95.97 ± 2.03,88.11 ± 1.53,7.86 ± 2.41,92.29 ± 2.02,85.56 ± 2.05,6.73 ± 1.50
9,멀티모달,pca,5,95.96 ± 2.35,91.41 ± 2.07,4.54 ± 1.96,94.45 ± 1.81,89.37 ± 2.23,5.08 ± 2.10


### 2-2. 보고용 요약 — Target TEST

`Bias` = `Tgt-Val − Tgt-Test`. 양수면 val 로 고른 탓에 낙관적으로 부풀었다는 뜻.

In [6]:
num = agg_num(sweep, ["model", "method"])

report = pd.DataFrame({
    "n":         num[("n", "")],
    "Tgt-Test":  num[("target_test_acc", "mean")],
    "±":         num[("target_test_acc", "std")],
    "Tgt-Val":   num[("target_acc", "mean")],
    "± ":        num[("target_acc", "std")],
    "Bias(V−T)": (num[("target_acc", "mean")] - num[("target_test_acc", "mean")]).round(2),
    "Src-Test":  num[("source_test_acc", "mean")],
    "Shift-Test": num[("test_shift", "mean")],
}).reset_index()
report

,model,method,n,Tgt-Test,±,Tgt-Val,±,Bias(V−T),Src-Test,Shift-Test
0,IMU 단독,raw,5,21.81,2.73,21.36,2.37,-0.45,82.01,60.19
1,IMU 단독,permutation,5,74.22,2.30,75.85,2.68,1.63,88.13,13.92
2,IMU 단독,gravity,5,70.05,6.14,73.79,4.57,3.74,88.07,18.02
3,IMU 단독,kabsch,5,63.78,5.13,62.64,4.34,-1.14,88.79,25.01
4,IMU 단독,pca,5,75.00,2.01,75.36,3.06,0.36,90.66,15.66
5,멀티모달,raw,5,31.76,1.73,35.29,3.69,3.53,87.78,56.03
6,멀티모달,permutation,5,87.45,2.63,89.91,0.81,2.46,92.26,4.82
7,멀티모달,gravity,5,88.75,1.98,91.97,1.68,3.22,92.90,4.15
8,멀티모달,kabsch,5,85.56,2.05,88.11,1.53,2.55,92.29,6.73
9,멀티모달,pca,5,89.37,2.23,91.41,2.07,2.04,94.45,5.08


In [7]:
# 방법을 행, 모델을 열로 놓은 최종 비교 표 (논문 표 형태)
pivot = (sweep.groupby(["method", "model"], observed=True)["target_test_acc"]
              .agg(["mean", "std"]).round(2))
pivot = pivot.assign(cell=lambda d: d["mean"].map("{:.2f}".format) + " ± " + d["std"].map("{:.2f}".format))
pivot_tgt = pivot["cell"].unstack("model")
pivot_tgt.columns.name = "Target TEST acc (%)"
pivot_tgt

Target TEST acc (%),IMU 단독,멀티모달
method,,
raw,21.81 ± 2.73,31.76 ± 1.73
permutation,74.22 ± 2.30,87.45 ± 2.63
gravity,70.05 ± 6.14,88.75 ± 1.98
kabsch,63.78 ± 5.13,85.56 ± 2.05
pca,75.00 ± 2.01,89.37 ± 2.23


### 2-3. seed 별 전체 값

In [8]:
per_seed = (sweep.sort_values(["model", "method", "seed"])
                 [["model", "method", "seed", *METRICS]]
                 .rename(columns=METRICS)
                 .reset_index(drop=True))
per_seed.style.format({c: "{:.2f}" for c in METRICS.values()}) \
    .background_gradient(subset=["Tgt-Test"], cmap="RdYlGn", vmin=20, vmax=95)

,model,method,seed,Src-Val,Tgt-Val,Shift-Val,Src-Test,Tgt-Test,Shift-Test
0,IMU 단독,raw,0,89.86,21.82,68.04,82.43,22.93,59.50
1,IMU 단독,raw,1,86.63,22.95,63.68,84.87,23.69,61.18
2,IMU 단독,raw,2,86.02,24.05,61.96,76.75,23.87,52.88
3,IMU 단독,raw,3,90.97,19.78,71.20,86.94,21.29,65.66
4,IMU 단독,raw,4,85.13,18.19,66.93,79.03,17.29,61.74
5,IMU 단독,permutation,0,88.52,80.16,8.37,89.97,77.75,12.22
6,IMU 단독,permutation,1,93.65,73.88,19.77,89.76,73.40,16.36
7,IMU 단독,permutation,2,85.91,73.39,12.52,83.76,74.37,9.39
8,IMU 단독,permutation,3,90.92,76.21,14.71,88.27,71.37,16.90
9,IMU 단독,permutation,4,90.14,75.62,14.52,88.91,74.19,14.71


In [9]:
# seed 간 퍼짐 — 방법 간 차이가 이 noise floor 를 넘는지 보는 용도
spread = (sweep.groupby(["model", "method"], observed=True)["target_test_acc"]
               .agg(min="min", max="max", ptp=lambda s: s.max() - s.min(), std="std")
               .round(2).reset_index())
spread

,model,method,min,max,ptp,std
0,IMU 단독,raw,17.29,23.87,6.58,2.73
1,IMU 단독,permutation,71.37,77.75,6.37,2.30
2,IMU 단독,gravity,64.09,77.45,13.36,6.14
3,IMU 단독,kabsch,56.58,71.08,14.50,5.13
4,IMU 단독,pca,72.72,76.69,3.96,2.01
5,멀티모달,raw,29.86,34.29,4.43,1.73
6,멀티모달,permutation,82.97,89.72,6.75,2.63
7,멀티모달,gravity,85.82,90.84,5.02,1.98
8,멀티모달,kabsch,82.94,87.67,4.73,2.05
9,멀티모달,pca,86.35,92.48,6.14,2.23


## 3. Learnable R — 같은 test 규약 (2026-07-23 스윕)

학습되는 R(SO(3))이 고정 R 을 넘는지 본다. 데이터는 `preprocessed_MM_raw_isotropic`
(축별 표준화를 하면 R 이 복원해야 할 축 정보가 지워지므로 등방 스케일을 쓴다),
세션 분할은 위 5방법과 **동일**하다.

`join<N>` = align-first 의 target 합류 지연 epoch (0=지연 없음=순수 JOINT).
`gravonly`/`pcaonly` = 기하 prior 손실 ablation(@join10).
`join10_ep30` = 5방법 표와 epoch 을 맞춘 대조군 — 나머지는 60ep 이라 그대로 비교하면
"R 이 좋아서"인지 "2배 더 돌아서"인지 구분되지 않는다.

In [10]:
LEARN = {
    "IMU 단독": ("results/Learnable_R/learnable_r_cdan_alignfirst_result_{}.json",
                ["join0", "join5", "join10", "join15", "join20", "join30",
                 "gravonly", "pcaonly", "join10_ep30"]),
    "멀티모달":  ("results/Learnable_R/learnable_r_mm_alignfirst_result_{}.json",
                ["join0"]),
}
lk = {tmpl.format(t): (label, t)
      for label, (tmpl, tags) in LEARN.items() for t in tags}
order = list(dict.fromkeys(t for _, tags in LEARN.values() for t in tags))

# 옛 2분할 결과가 같은 파일명으로 남아 있을 수 있으므로 protocol 로 한 번 더 거른다
learn = df_runs[df_runs.file.isin(lk) & (df_runs.protocol == "train/val/test")].copy()

if learn.empty:
    print("learnable 결과(신 규약) 없음 — run_learnable_sweep_2026-07-23.sh 실행 중이거나 미실행")
    tbl_learn = None
else:
    learn["model"]  = pd.Categorical(learn.file.map(lambda f: lk[f][0]), list(LEARN), ordered=True)
    learn["config"] = pd.Categorical(learn.file.map(lambda f: lk[f][1]), order, ordered=True)
    done = set(learn.file)
    todo = [lk[f][0] + " | " + lk[f][1] for f in lk if f not in done]
    if todo:
        print("아직 안 끝난 잡:", ", ".join(sorted(todo)))
    tbl_learn = agg_pm(learn, ["model", "config"])
tbl_learn

,model,config,n,Src-Val,Tgt-Val,Shift-Val,Src-Test,Tgt-Test,Shift-Test
0,IMU 단독,join0,5,86.30 ± 3.16,81.29 ± 5.62,5.01 ± 3.37,81.79 ± 4.17,78.19 ± 5.89,3.60 ± 5.38
1,IMU 단독,join5,5,87.89 ± 1.27,84.11 ± 4.01,3.78 ± 4.25,85.79 ± 2.50,81.24 ± 2.72,4.54 ± 4.04
2,IMU 단독,join10,5,88.21 ± 1.69,83.68 ± 4.57,4.53 ± 5.67,86.10 ± 3.51,80.08 ± 5.03,6.02 ± 8.17
3,IMU 단독,join15,5,88.60 ± 2.16,85.03 ± 4.09,3.57 ± 5.76,83.48 ± 4.51,81.76 ± 4.48,1.72 ± 8.41
4,IMU 단독,join20,5,86.93 ± 2.00,85.92 ± 3.91,1.01 ± 4.75,83.63 ± 3.57,82.62 ± 3.68,1.01 ± 4.79
5,IMU 단독,join30,5,86.55 ± 1.52,73.33 ± 10.72,13.22 ± 10.82,84.29 ± 3.90,70.95 ± 9.78,13.34 ± 12.37
6,IMU 단독,gravonly,5,91.26 ± 5.13,32.93 ± 11.36,58.33 ± 9.16,87.13 ± 4.74,32.82 ± 11.54,54.31 ± 10.42
7,IMU 단독,pcaonly,5,88.08 ± 2.14,81.55 ± 3.67,6.53 ± 3.76,83.82 ± 1.11,79.46 ± 3.55,4.36 ± 3.28
8,IMU 단독,join10_ep30,5,84.47 ± 1.83,82.14 ± 4.90,2.33 ± 5.75,81.15 ± 3.95,78.60 ± 3.86,2.54 ± 5.26
9,멀티모달,join0,5,91.40 ± 5.57,90.82 ± 2.44,0.58 ± 4.43,86.45 ± 7.41,81.94 ± 2.41,4.51 ± 5.65


### 3-1. 고정 R vs 학습 R (Target TEST)

같은 분할·같은 selection 규약이므로 두 수치는 직접 비교된다.
차이가 seed std 안에 들어가면 "동급"이지 우열이 아니다.

In [11]:
if tbl_learn is None:
    cmp_fixed_learn = None
    print("learnable 결과 대기 중")
else:
    fx = agg_num(sweep, ["model", "method"])
    ln = agg_num(learn, ["model", "config"])
    fixed_best = fx[("target_test_acc", "mean")].groupby(level="model", observed=True).idxmax()

    rows = []
    for model, key in fixed_best.items():
        f_mu, f_sd = fx.loc[key, ("target_test_acc", "mean")], fx.loc[key, ("target_test_acc", "std")]
        sub = ln.loc[model] if model in ln.index.get_level_values(0) else None
        if sub is None or sub.empty:
            continue
        best_cfg = sub[("target_test_acc", "mean")].idxmax()
        l_mu, l_sd = sub.loc[best_cfg, ("target_test_acc", "mean")], sub.loc[best_cfg, ("target_test_acc", "std")]
        row = {
            "model": model,
            "고정 R (best)": f"{key[1]}: {f_mu:.2f} ± {f_sd:.2f}",
            "학습 R (best)": f"{best_cfg}: {l_mu:.2f} ± {l_sd:.2f}",
            "차이(학습−고정)": round(l_mu - f_mu, 2),
            "seed std 이내?": "예 (동급)" if abs(l_mu - f_mu) < max(f_sd, l_sd) else "아니오",
        }
        # 60ep 짜리 best 는 5방법 표(30ep)와 epoch 이 다르다. 정합 대조군이 있으면 같이 싣는다.
        if ("join10_ep30" in sub.index) and pd.notna(sub.loc["join10_ep30", ("target_test_acc", "mean")]):
            e_mu = sub.loc["join10_ep30", ("target_test_acc", "mean")]
            e_sd = sub.loc["join10_ep30", ("target_test_acc", "std")]
            row["학습 R (epoch 정합)"] = f"join10_ep30: {e_mu:.2f} ± {e_sd:.2f}"
            row["차이(정합−고정)"] = round(e_mu - f_mu, 2)
        rows.append(row)
    cmp_fixed_learn = pd.DataFrame(rows)
    display(cmp_fixed_learn)

,model,고정 R (best),학습 R (best),차이(학습−고정),seed std 이내?,학습 R (epoch 정합),차이(정합−고정)
0,IMU 단독,pca: 75.00 ± 2.01,join20: 82.62 ± 3.68,7.62,아니오,join10_ep30: 78.60 ± 3.86,3.6
1,멀티모달,pca: 89.37 ± 2.23,join0: 81.94 ± 2.41,-7.43,아니오,NaN,NaN


### 3-2. 보고용 요약 — Target TEST

2-2 와 같은 형식. `Bias` 가 크면 val 로 고른 탓에 부풀었다는 뜻이라,
config 간 비교를 Tgt-Val 로 하면 안 된다는 신호다.

In [12]:
if tbl_learn is None:
    learn_report = learn_pivot = None
else:
    lnum = agg_num(learn, ["model", "config"])
    learn_report = pd.DataFrame({
        "n":          lnum[("n", "")],
        "Tgt-Test":   lnum[("target_test_acc", "mean")],
        "±":          lnum[("target_test_acc", "std")],
        "Tgt-Val":    lnum[("target_acc", "mean")],
        "± ":         lnum[("target_acc", "std")],
        "Bias(V−T)":  (lnum[("target_acc", "mean")] - lnum[("target_test_acc", "mean")]).round(2),
        "Src-Test":   lnum[("source_test_acc", "mean")],
        "Shift-Test": lnum[("test_shift", "mean")],
    }).reset_index()

    # config 를 행, 모델을 열로 (5방법 표의 pivot_tgt 와 같은 모양)
    lp = (learn.groupby(["config", "model"], observed=True)["target_test_acc"]
               .agg(["mean", "std"]).round(2))
    learn_pivot = (lp.assign(cell=lambda d: d["mean"].map("{:.2f}".format)
                                            + " ± " + d["std"].map("{:.2f}".format))
                     ["cell"].unstack("model").fillna("—"))
    learn_pivot.columns.name = "Target TEST acc (%)"
learn_report

,model,config,n,Tgt-Test,±,Tgt-Val,±,Bias(V−T),Src-Test,Shift-Test
0,IMU 단독,join0,5,78.19,5.89,81.29,5.62,3.10,81.79,3.60
1,IMU 단독,join5,5,81.24,2.72,84.11,4.01,2.87,85.79,4.54
2,IMU 단독,join10,5,80.08,5.03,83.68,4.57,3.60,86.10,6.02
3,IMU 단독,join15,5,81.76,4.48,85.03,4.09,3.27,83.48,1.72
4,IMU 단독,join20,5,82.62,3.68,85.92,3.91,3.30,83.63,1.01
5,IMU 단독,join30,5,70.95,9.78,73.33,10.72,2.38,84.29,13.34
6,IMU 단독,gravonly,5,32.82,11.54,32.93,11.36,0.11,87.13,54.31
7,IMU 단독,pcaonly,5,79.46,3.55,81.55,3.67,2.09,83.82,4.36
8,IMU 단독,join10_ep30,5,78.60,3.86,82.14,4.90,3.54,81.15,2.54
9,멀티모달,join0,5,81.94,2.41,90.82,2.44,8.88,86.45,4.51


### 3-3. join 지연 효과 — seed-paired 차이

config 별 mean ± std 만 보면 seed 간 퍼짐(±3~6)에 묻혀 join 곡선의 봉우리가
진짜인지 알 수 없다. 같은 seed 끼리 짝지어 빼면 seed 자체의 변동이 상쇄된다.
기준은 `join0`(지연 없음 = 순수 JOINT).

`5 seed 전부 양수?` 가 `예` 여야 부호가 확실한 것이고, 그렇지 않으면
평균 차이가 아무리 커 보여도 "구분 안 됨"으로 읽어야 한다.

In [13]:
BASE = "join0"

if tbl_learn is None:
    learn_paired = None
else:
    rows = []
    for model, sub in learn.groupby("model", observed=True):
        wide = sub.pivot(index="seed", columns="config", values="target_test_acc")
        if BASE not in wide or wide[BASE].isna().all():
            continue
        for cfg in [c for c in order if c in wide and c != BASE]:
            d = (wide[cfg] - wide[BASE]).dropna()
            if d.empty:
                continue
            rows.append({
                "model": model, "config": cfg, f"vs {BASE}": "Δ Tgt-Test",
                "mean Δ": round(d.mean(), 2),
                "std Δ":  round(d.std(), 2),
                "min Δ":  round(d.min(), 2),
                "max Δ":  round(d.max(), 2),
                f"{len(d)} seed 전부 양수?": "예" if (d > 0).all() else "아니오",
            })
    learn_paired = pd.DataFrame(rows)
learn_paired

,model,config,vs join0,mean Δ,std Δ,min Δ,max Δ,5 seed 전부 양수?
0,IMU 단독,join5,Δ Tgt-Test,3.05,4.33,-0.53,9.60,아니오
1,IMU 단독,join10,Δ Tgt-Test,1.89,8.08,-7.37,14.56,아니오
2,IMU 단독,join15,Δ Tgt-Test,3.57,5.82,-3.96,12.04,아니오
3,IMU 단독,join20,Δ Tgt-Test,4.43,3.01,1.09,8.75,예
4,IMU 단독,join30,Δ Tgt-Test,-7.24,10.20,-22.84,1.15,아니오
5,IMU 단독,gravonly,Δ Tgt-Test,-45.37,11.01,-56.08,-33.00,아니오
6,IMU 단독,pcaonly,Δ Tgt-Test,1.27,5.45,-2.70,10.75,아니오
7,IMU 단독,join10_ep30,Δ Tgt-Test,0.41,5.48,-5.58,8.37,아니오


### 3-4. seed 별 전체 값

In [14]:
if tbl_learn is None:
    learn_per_seed = None
else:
    learn_per_seed = (learn.sort_values(["model", "config", "seed"])
                           [["model", "config", "seed", *METRICS]]
                           .rename(columns=METRICS)
                           .reset_index(drop=True))
    display(learn_per_seed.style.format({c: "{:.2f}" for c in METRICS.values()})
            .background_gradient(subset=["Tgt-Test"], cmap="RdYlGn", vmin=20, vmax=95))

,model,config,seed,Src-Val,Tgt-Val,Shift-Val,Src-Test,Tgt-Test,Shift-Test
0,IMU 단독,join0,0,80.78,72.19,8.59,76.17,69.76,6.41
1,IMU 단독,join0,1,88.19,79.44,8.75,82.54,75.25,7.29
2,IMU 단독,join0,2,86.46,84.46,2.00,84.55,79.51,5.05
3,IMU 단독,join0,3,88.13,84.91,3.22,86.57,81.42,5.16
4,IMU 단독,join0,4,87.91,85.43,2.48,79.14,85.03,-5.89
5,IMU 단독,join5,0,87.86,80.80,7.05,85.40,79.36,6.04
6,IMU 단독,join5,1,89.86,83.55,6.31,82.48,80.56,1.92
7,IMU 단독,join5,2,86.46,80.84,5.63,89.49,79.45,10.04
8,IMU 단독,join5,3,88.08,84.78,3.29,86.09,80.89,5.21
9,IMU 단독,join5,4,87.19,90.58,-3.39,85.46,85.97,-0.51


## 4. 전체 인벤토리

`results/` 아래 모든 multi-seed 결과. 옛 2분할 규약 파일도 포함하되 `protocol` 로 구분한다.

In [15]:
inventory = agg_pm(df_runs, ["group", "protocol", "file", "tag", "modality"])
inventory = inventory.merge(
    df_runs.groupby("file", as_index=False)["mtime"].max(), on="file", how="left")
inventory.sort_values(["group", "mtime"], ascending=[True, False]).reset_index(drop=True)

,group,protocol,file,tag,modality,n,Src-Val,Tgt-Val,Shift-Val,Src-Test,Tgt-Test,Shift-Test,mtime
0,IMU,train/val/test,results/IMU/imu_cdan_result_pca.json,pca,imu_only,5,93.64 ± 1.90,75.36 ± 3.06,18.27 ± 3.93,90.66 ± 1.97,75.00 ± 2.01,15.66 ± 1.52,2026-07-22 17:28
1,IMU,train/val/test,results/IMU/imu_cdan_result_kabsch.json,kabsch,imu_only,5,91.11 ± 1.98,62.64 ± 4.34,28.47 ± 5.19,88.79 ± 0.85,63.78 ± 5.13,25.01 ± 4.79,2026-07-22 16:48
2,IMU,train/val/test,results/IMU/imu_cdan_result_gravity.json,gravity,imu_only,5,90.51 ± 3.36,73.79 ± 4.57,16.72 ± 4.80,88.07 ± 4.08,70.05 ± 6.14,18.02 ± 4.42,2026-07-22 16:08
3,IMU,train/val/test,results/IMU/imu_cdan_result_permutation.json,permutation,imu_only,5,89.83 ± 2.87,75.85 ± 2.68,13.98 ± 4.12,88.13 ± 2.54,74.22 ± 2.30,13.92 ± 3.12,2026-07-22 15:28
4,IMU,train/val/test,results/IMU/imu_cdan_result_raw.json,raw,imu_only,5,87.72 ± 2.55,21.36 ± 2.37,66.36 ± 3.64,82.01 ± 4.16,21.81 ± 2.73,60.19 ± 4.67,2026-07-22 14:48
5,Learnable_R,train/val/test,results/Learnable_R/learnable_r_cdan_alignfirs...,g1.0_p1.0,imu_only,5,86.85 ± 1.43,85.89 ± 3.15,0.97 ± 4.22,84.70 ± 4.67,80.45 ± 4.41,4.25 ± 7.88,2026-07-25 16:20
6,Learnable_R,train/val/test,results/Learnable_R/learnable_r_cdan_alignfirs...,g1.0_p0.8,imu_only,5,85.34 ± 4.31,84.34 ± 5.52,1.00 ± 3.09,82.61 ± 4.70,79.84 ± 4.53,2.78 ± 5.19,2026-07-25 15:46
7,Learnable_R,train/val/test,results/Learnable_R/learnable_r_cdan_alignfirs...,g1.0_p0.6,imu_only,5,86.79 ± 4.10,79.66 ± 6.26,7.12 ± 9.52,86.05 ± 5.19,75.53 ± 5.59,10.53 ± 10.24,2026-07-25 15:13
8,Learnable_R,train/val/test,results/Learnable_R/learnable_r_cdan_alignfirs...,g1.0_p0.4,imu_only,5,89.40 ± 1.53,81.15 ± 5.99,8.25 ± 6.00,86.97 ± 2.90,77.72 ± 4.69,9.25 ± 6.94,2026-07-25 14:39
9,Learnable_R,train/val/test,results/Learnable_R/learnable_r_cdan_alignfirs...,g1.0_p0.2,imu_only,5,91.67 ± 1.96,78.26 ± 7.82,13.40 ± 8.09,89.97 ± 5.29,77.09 ± 4.98,12.87 ± 8.98,2026-07-25 14:05


## 5. 내보내기 — CSV / 마크다운

In [16]:
OUT = ROOT / "results" / "tables"
OUT.mkdir(parents=True, exist_ok=True)
STAMP = "2026-07-22_method_sweep"

per_seed.to_csv(OUT / f"{STAMP}_per_seed.csv", index=False, encoding="utf-8-sig")
report.to_csv(OUT / f"{STAMP}_summary.csv", index=False, encoding="utf-8-sig")
inventory.to_csv(OUT / "all_results_inventory.csv", index=False, encoding="utf-8-sig")

md_txt = "\n\n".join([
    "### Target TEST accuracy (%, mean ± std over 5 seeds)", to_markdown(pivot_tgt.reset_index()),
    "### 전체 지표", to_markdown(tbl_full),
    "### seed 별", to_markdown(per_seed),
])
(OUT / f"{STAMP}.md").write_text(md_txt, encoding="utf-8")

for p in sorted(OUT.iterdir()):
    print(p.relative_to(ROOT))

results/tables/2026-07-22_method_sweep.md
results/tables/2026-07-22_method_sweep_per_seed.csv
results/tables/2026-07-22_method_sweep_summary.csv
results/tables/2026-07-23_learnable_r_sweep.md
results/tables/2026-07-23_learnable_r_sweep_paired.csv
results/tables/2026-07-23_learnable_r_sweep_per_seed.csv
results/tables/2026-07-23_learnable_r_sweep_summary.csv
results/tables/all_results_inventory.csv


In [17]:
STAMP_L = "2026-07-23_learnable_r_sweep"

if tbl_learn is None:
    print("learnable 결과 없음 — 내보낼 것이 없다")
else:
    learn_per_seed.to_csv(OUT / f"{STAMP_L}_per_seed.csv", index=False, encoding="utf-8-sig")
    learn_report.to_csv(OUT / f"{STAMP_L}_summary.csv", index=False, encoding="utf-8-sig")
    learn_paired.to_csv(OUT / f"{STAMP_L}_paired.csv", index=False, encoding="utf-8-sig")

    header = (
        "# Learnable R 스윕 — Target TEST (2026-07-23)\n\n"
        "학습되는 R(SO(3)) 이 고정 R 을 넘는지. 데이터 `preprocessed_MM_raw_isotropic`,\n"
        "세션 분할은 [2026-07-22 5방법 표](2026-07-22_method_sweep.md) 와 동일하므로\n"
        "두 표의 수치는 직접 비교된다.\n\n"
        "**규약**: 세션 단위 층화 7:1.5:1.5. model selection = target **val**(oracle),\n"
        "보고 수치 = target **TEST**. seed 0–4.\n\n"
        "`join<N>` = align-first 의 target 합류 지연 epoch (0 = 지연 없음 = 순수 JOINT).\n"
        "`gravonly`/`pcaonly` = 기하 prior 손실 ablation (@join10).\n"
        "`join10_ep30` = 5방법 표(30ep)와 epoch 을 맞춘 대조군 — 나머지는 60ep 이라\n"
        "그대로 비교하면 R 의 기여와 학습량의 기여가 섞인다."
    )
    md_l = "\n\n".join([
        header,
        "## Target TEST accuracy (%, mean ± std over 5 seeds)", to_markdown(learn_pivot.reset_index()),
        "## 고정 R vs 학습 R", to_markdown(cmp_fixed_learn),
        "## 전체 지표", to_markdown(tbl_learn),
        f"## join 지연의 seed-paired 효과 (기준 {BASE})", to_markdown(learn_paired),
        "## seed 별", to_markdown(learn_per_seed),
    ])
    (OUT / f"{STAMP_L}.md").write_text(md_l, encoding="utf-8")

for p in sorted(OUT.iterdir()):
    print(p.relative_to(ROOT))

results/tables/2026-07-22_method_sweep.md
results/tables/2026-07-22_method_sweep_per_seed.csv
results/tables/2026-07-22_method_sweep_summary.csv
results/tables/2026-07-23_learnable_r_sweep.md
results/tables/2026-07-23_learnable_r_sweep_paired.csv
results/tables/2026-07-23_learnable_r_sweep_per_seed.csv
results/tables/2026-07-23_learnable_r_sweep_summary.csv
results/tables/all_results_inventory.csv


In [18]:
print(md_txt[:1200])

### Target TEST accuracy (%, mean ± std over 5 seeds)

| method | IMU 단독 | 멀티모달 |
|---|---|---|
| raw | 21.81 ± 2.73 | 31.76 ± 1.73 |
| permutation | 74.22 ± 2.30 | 87.45 ± 2.63 |
| gravity | 70.05 ± 6.14 | 88.75 ± 1.98 |
| kabsch | 63.78 ± 5.13 | 85.56 ± 2.05 |
| pca | 75.00 ± 2.01 | 89.37 ± 2.23 |

### 전체 지표

| model | method | n | Src-Val | Tgt-Val | Shift-Val | Src-Test | Tgt-Test | Shift-Test |
|---|---|---|---|---|---|---|---|---|
| IMU 단독 | raw | 5 | 87.72 ± 2.55 | 21.36 ± 2.37 | 66.36 ± 3.64 | 82.01 ± 4.16 | 21.81 ± 2.73 | 60.19 ± 4.67 |
| IMU 단독 | permutation | 5 | 89.83 ± 2.87 | 75.85 ± 2.68 | 13.98 ± 4.12 | 88.13 ± 2.54 | 74.22 ± 2.30 | 13.92 ± 3.12 |
| IMU 단독 | gravity | 5 | 90.51 ± 3.36 | 73.79 ± 4.57 | 16.72 ± 4.80 | 88.07 ± 4.08 | 70.05 ± 6.14 | 18.02 ± 4.42 |
| IMU 단독 | kabsch | 5 | 91.11 ± 1.98 | 62.64 ± 4.34 | 28.47 ± 5.19 | 88.79 ± 0.85 | 63.78 ± 5.13 | 25.01 ± 4.79 |
| IMU 단독 | pca | 5 | 93.64 ± 1.90 | 75.36 ± 3.06 | 18.27 ± 3.93 | 90.66 ± 1.97 | 75.00 ± 2.01 | 15.6